In [1]:
import pytensorlab as tl
import numpy as np
import time
from tucker_tensor import TuckerDecomposition, SparseTupleTensor

2025-12-11 11:05:00.952290: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
pop_tensor = SparseTupleTensor.load_from_disk("fineweb_large", "sc", 1000, map_location="cpu")
tucker_tensor = TuckerDecomposition.load_from_disk("fineweb_large", "sc", 1000, 150, 2500, map_location="cpu")

Loaded Tucker decomposition with the following parameters:
  timestamp: 2025-12-05T11:58:08.497162Z
  dataset: fineweb_dutch
  input_vectors: 10000000
  method: sc
  dimensionality: 1000
  rank: [150, 150, 150]
  max_iterations: 2500
  iterations: 1613
  final_error: 0.9411253929138184
  tolerance: 1e-07
  random_state: 42
  output_tensor: sc_1000d_150r_2500i.pt
  runtime_seconds: 1785.31


In [3]:
tl_sparse = tl.SparseTensor(data=pop_tensor.tensor.values(),
                            indices=pop_tensor.tensor.indices(),
                            shape=pop_tensor.tensor.shape)

In [4]:
tl_tucker = tl.TuckerTensor(core=tucker_tensor.core,
                            factors=tucker_tensor.factors)

In [5]:
tl_tucker

Tucker tensor of shape (1000, 1000, 1000) with core shape (150, 150, 150)

In [6]:
tl_tucker.core.shape

(150, 150, 150)

In [7]:
start = time.time()
res = tl_sparse.matricize(1)
end = time.time()
print(f"Matricization time SparseTensor: {end - start} seconds")
res

Matricization time SparseTensor: 0.007672309875488281 seconds


<COOrdinate sparse matrix of dtype 'float32'
	with 754107 stored elements and shape (1000, 1000000)>

In [33]:
import os, multiprocessing
from threadpoolctl import threadpool_limits  # NEW
import tensorly_custom as tlc
tlc.set_backend('numpy')

max_cpu_frac = 0.5
def get_eval_num_threads(fraction: float = 0.75, min_threads: int = 1) -> int:
    """Return n_threads ≈ fraction * available CPUs (at least min_threads)."""
    try:
        n_cores = multiprocessing.cpu_count()
    except NotImplementedError:
        n_cores = os.cpu_count() or 1

    n_threads = max(min_threads, int(n_cores * fraction))
    return n_threads
n_threads = get_eval_num_threads(fraction=max_cpu_frac, min_threads=1)

In [43]:
with threadpool_limits(1):
    start = time.time()
    tl_reconstruct = tl.tmprod(tl_tucker.core, tl_tucker.factors, range(tl_tucker.ndim))
    print(f"Reconstruction time TuckerTensor: {time.time() - start} seconds")

Reconstruction time TuckerTensor: 3.433307647705078 seconds


In [45]:
with threadpool_limits(1):
    start = time.time()
    tlc_reconstruct = tlc.tucker_to_tensor((tl_tucker.core, tl_tucker.factors))
    print(f"Reconstruction time TuckerTensor: {time.time() - start} seconds")

Reconstruction time TuckerTensor: 4.194591045379639 seconds


In [50]:
tl_factor_0 = tl.tmprod(tl_tucker.core, tl_tucker.factors[1:], [1, 2])
tl_factor_0.shape

(150, 1000, 1000)

In [53]:
tlc_factor_0 = tlc.tucker_to_tensor((tl_tucker.core, tl_tucker.factors), skip_factor=0)
tlc_factor_0.shape

(150, 1000, 1000)

In [54]:
np.allclose(tl_factor_0, tlc_factor_0)

True

In [55]:
for mode in range(tl_tucker.ndim):
    tl_factor = tl.tmprod(tl_tucker.core, [f for i, f in enumerate(tl_tucker.factors) if i != mode],
                         [i for i in range(tl_tucker.ndim) if i != mode])
    tlc_factor = tlc.tucker_to_tensor((tl_tucker.core, tl_tucker.factors), skip_factor=mode)
    assert np.allclose(tl_factor, tlc_factor)

# testing out their decomposition
Seems to work very well on small toy examples.
Extremely small even on our super small tensors.

In [ ]:
input_tensor = tl_sparse
tl.lmlra(input_tensor, (150, 150, 150))

Step 1: Initialization is mlsvd... 

In [ ]:
2+2